In [62]:
import pandas as pd
df = pd.read_csv("C:/Users/vldma/Datathon/Datasets/Modelling datasets/fe_features_regression.csv")
df.head()

,segment_id,carretera,comunidad_autonoma,provincia,municipio,length_m,road_total_refill_points,road_mean_power_kw,road_n_operators,road_n_segments,road_n_gaps,road_mean_nearest_m,road_max_nearest_m,road_centroid_lat,road_centroid_lon,road_min_lat,road_min_lon,road_span_lat,road_span_lon,nearest_grid_dist_deg,grid_capacity_occupied_mw,grid_capacity_pending_mw,max_voltage_kv,grid_capacity_3nearest_mw,grid_maxvoltage_3nearest_kv,road_type_rank,road_refill_per_100km,afir_power_gap_kw,afir_power_compliant,power_range_kw,has_hpc,connector_diversity,refill_per_station,coverage_ratio,gap_severity,grid_headroom_mw,grid_strong_hv,grid_capacity_per_station,log_length_m,log_road_max_nearest_m,log_road_mean_nearest_m,log_road_refill_per_100km,demand_vs_supply,grid_x_power,tipo_de_via_Autopista peaje,tipo_de_via_Carretera convencional,tipo_de_via_Multicarril,comunidad_autonoma_02 - Aragón,comunidad_autonoma_04 - Illes Balears,comunidad_autonoma_05 - Canarias,comunidad_autonoma_07 - C.León,comunidad_autonoma_08 - C.La Mancha,comunidad_autonoma_09 - Catalunya,comunidad_autonoma_11 - Extremadura,comunidad_autonoma_12 - Galicia,comunidad_autonoma_15 - Navarra,comunidad_autonoma_unknown,stations_deficit_afir
0,0,TO-23,04 - Illes Balears,Illes Balears,Andratx,7527.979245,45,83.450000,7,1.0,0.0,74.822269,74.822269,39.864278,-3.972001,39.862125,-3.980628,0.007115,0.021446,0.508059,74.26,0.00,15.0,169.22,15.0,2,597.769980,0.0,1,166.550000,1,2,3.750000,1.0,1.000000,94.96,0,14.101667,8.926515,4.328392,4.328392,6.394878,0.055650,23740.0,0,0,1,0,1,0,0,0,0,0,0,0,0,0
1,1,AP-7S,01 - Andalucía,Málaga,Marbella,22078.044641,4,27.333333,2,5.0,0.0,3104.503662,4269.023990,36.546676,-4.760377,36.414670,-5.225154,0.198497,0.697844,0.048716,46.33,0.01,66.0,146.49,66.0,3,4.441374,120.0,0,2.666667,0,2,1.333333,1.0,1.375107,100.16,0,48.830000,10.002384,8.359375,8.040931,1.694032,2.309470,3004.8,1,0,0,0,0,0,0,0,0,0,0,0,0,0
2,2,AP-7S,01 - Andalucía,Málaga,Marbella,24146.013100,4,27.333333,2,5.0,0.0,3104.503662,4269.023990,36.546676,-4.760377,36.414670,-5.225154,0.198497,0.697844,0.048716,46.33,0.01,66.0,146.49,66.0,3,4.441374,120.0,0,2.666667,0,2,1.333333,1.0,1.375107,100.16,0,48.830000,10.091916,8.359375,8.040931,1.694032,2.330142,3004.8,1,0,0,0,0,0,0,0,0,0,0,0,0,0
3,3,AP-7S,01 - Andalucía,Málaga,Marbella,28323.852697,4,27.333333,2,5.0,0.0,3104.503662,4269.023990,36.546676,-4.760377,36.414670,-5.225154,0.198497,0.697844,0.048716,46.33,0.01,66.0,146.49,66.0,3,4.441374,120.0,0,2.666667,0,2,1.333333,1.0,1.375107,100.16,0,48.830000,10.251495,8.359375,8.040931,1.694032,2.366988,3004.8,1,0,0,0,0,0,0,0,0,0,0,0,0,0
4,4,AP-7S,01 - Andalucía,Málaga,Marbella,11492.490960,4,27.333333,2,5.0,0.0,3104.503662,4269.023990,36.546676,-4.760377,36.414670,-5.225154,0.198497,0.697844,0.048716,46.33,0.01,66.0,146.49,66.0,3,4.441374,120.0,0,2.666667,0,2,1.333333,1.0,1.375107,100.16,0,48.830000,9.349536,8.359375,8.040931,1.694032,2.158733,3004.8,1,0,0,0,0,0,0,0,0,0,0,0,0,0


# Classification — Optimal EV Station Locations

Answers the **where?** question of the hybrid pipeline.

- **Input**: `fe_features_classification.csv` (produced by `FE.ipynb`)
- **Target**: `is_underserved` — `1` iff the segment's nearest charger is **> 20 km away** (AFIR 60 km ÷ 3, operational safety margin; see `FE.ipynb` §3).
- **Positive event**: segment is a **candidate location for a new EV charger**.
- **Output**: `optimal_locations.csv` — segments ranked by predicted probability of needing a station, with road centroid coordinates for mapping.

### Notebook map

1. **Setup** — imports, load FE output.
2. **Data preparation** — stratified train/test split.
3. **Model comparison** — LogReg, Random Forest, LightGBM, XGBoost (all with balanced-class handling).
4. **Hyperparameter tuning** — GridSearchCV on the winning model.
5. **Final evaluation** — ROC-AUC, PR-AUC, F1, confusion matrix, threshold tuning, feature importance.
6. **Predict optimal locations** — refit on all data, rank segments, export CSV.

---
## 1 · Setup

### 1.1 Imports & config

In [63]:
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    GridSearchCV,
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    f1_score,
    precision_recall_curve,
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Gradient-boosting libraries — both optional, install if missing:
#   pip install lightgbm xgboost
try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except ImportError:
    HAS_LGBM = False
    print("lightgbm not installed — run: pip install lightgbm")

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("xgboost not installed — run: pip install xgboost")

RANDOM_STATE = 42
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)

### 1.2 Load FE output (single-file, metadata included)

`FE.ipynb` now writes `segment_id`, `carretera`, `comunidad_autonoma`, `provincia`, `municipio` as leading metadata columns alongside the features. The classification notebook therefore loads **only** `fe_features_classification.csv` — no raw-CSV merge required.

- `segment_meta` — the metadata columns (preserved verbatim for output mapping).
- `y` — the target `is_underserved` (1 if segment > 10 km from nearest charger).
- `X` — the feature matrix (everything else).

In [64]:
DATA_DIR = "C:/Users/vldma/Datathon/Datasets/Modelling datasets"

# Single-file load — FE now preserves segment_id / carretera / admin strings as
# leading metadata columns so the classification notebook needs no other input.
fe = pd.read_csv(f"{DATA_DIR}/fe_features_classification.csv")

META_COLS = [c for c in ["segment_id", "carretera", "comunidad_autonoma", "provincia", "municipio"] if c in fe.columns]
TARGET = "is_underserved"

segment_meta = fe[META_COLS].copy()
y = fe[TARGET].astype(int)
X = fe.drop(columns=META_COLS + [TARGET])

print("X (features):", X.shape)
print("y positive rate:", float(y.mean().round(3)))
print("\nmetadata columns recovered:", META_COLS)
segment_meta.head()

X (features): (1524, 52)
y positive rate: 0.041

metadata columns recovered: ['segment_id', 'carretera', 'comunidad_autonoma', 'provincia', 'municipio']


,segment_id,carretera,comunidad_autonoma,provincia,municipio
0,0,TO-23,04 - Illes Balears,Illes Balears,Andratx
1,1,AP-7S,01 - Andalucía,Málaga,Marbella
2,2,AP-7S,01 - Andalucía,Málaga,Marbella
3,3,AP-7S,01 - Andalucía,Málaga,Marbella
4,4,AP-7S,01 - Andalucía,Málaga,Marbella


---
## 2 · Data preparation

### Why the positive class is only ~4 %

- EU **AFIR** requires fast chargers every ≤60 km on TEN-T core roads. On today's Spanish interurban network **no segment** has `nearest_station_m > 60 000 m` — the strict AFIR label is 0 positives out of 1,524 rows and cannot be trained.
- We therefore classify against an **operational threshold = AFIR ÷ 3 = 20 km** (industry planning safety margin, buffer for 2027 EV growth). That yields **63 positives / 1,524 segments ≈ 4.1 %** — thin but trainable.
- We manage that imbalance four ways:
  1. **Stratified 5-fold CV** keeps the 4 % ratio in every fold, so no fold is pathologically easy or hard.
  2. **`class_weight='balanced'`** (and `scale_pos_weight` for XGBoost) reweights the loss so the model can't trivially predict "always negative".
  3. **PR-AUC** is the headline metric for model selection — it's the honest ranking metric under imbalance.
  4. **§5.2 threshold tuning** sweeps the PR curve on refit probabilities and picks the F1-optimal cut-off (0.5 is almost never optimal under imbalance).
- **No train/test hold-out.** With only ~13 positives in a 20 % test slice, a single split is too noisy. All evaluation is via 5-fold CV on the full dataset.

### 2.1 Stratified 5-fold cross-validation

One `StratifiedKFold` object is created here and reused everywhere below (model comparison, tuning, evaluation). Every fold preserves the 4 % positive rate.

In [65]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# sanity-check fold composition
for i, (tr, va) in enumerate(cv.split(X, y), start=1):
    print(f"fold {i}: train n={len(tr):5d} pos={int(y.iloc[tr].sum()):3d} | "
          f"val n={len(va):4d} pos={int(y.iloc[va].sum()):2d}")

print(f"\nfull dataset: X={X.shape}, positive rate={y.mean():.3f}")

fold 1: train n= 1219 pos= 51 | val n= 305 pos=12
fold 2: train n= 1219 pos= 50 | val n= 305 pos=13
fold 3: train n= 1219 pos= 50 | val n= 305 pos=13
fold 4: train n= 1219 pos= 50 | val n= 305 pos=13
fold 5: train n= 1220 pos= 51 | val n= 304 pos=12

full dataset: X=(1524, 52), positive rate=0.041


---
## 3 · Model comparison (baselines)

Imbalance handled natively — **no SMOTE, no synthetic samples**:
- LogReg / Random Forest / LightGBM: `class_weight='balanced'`
- XGBoost: `scale_pos_weight = n_neg / n_pos`

The real 4 % positive distribution is preserved in both training and validation, so probability calibration stays intact and metrics are honest.

Metrics, in order of trust:
- **PR-AUC (average precision)** — the honest ranking metric under imbalance; used to pick the winner.
- **ROC-AUC** — global ranking quality, robust to imbalance.
- **F1 @ 0.5** — point metric at the default threshold, reported for context only. Expect it to look low: `class_weight='balanced'` shifts the probability distribution so 0.5 is rarely the F1-optimal cut. The real operating F1 is reported in §5.2 after threshold tuning on the held-out set.

### 3.1 Define candidates & evaluate with 5-fold CV

In [66]:
# Each model handles class imbalance with its own native mechanism.
pos, neg = int(y.sum()), int((1 - y).sum())
scale_pos_weight = neg / max(pos, 1)
print(f"class counts: pos={pos}, neg={neg}, scale_pos_weight={scale_pos_weight:.3f}")

candidates = {
    "logreg": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        )),
    ]),
    "random_forest": RandomForestClassifier(
        n_estimators=400,
        class_weight="balanced",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
}

if HAS_LGBM:
    candidates["lightgbm"] = LGBMClassifier(
        n_estimators=400,
        learning_rate=0.05,
        num_leaves=31,
        class_weight="balanced",
        n_jobs=-1,
        random_state=RANDOM_STATE,
        verbose=-1,
    )

if HAS_XGB:
    candidates["xgboost"] = XGBClassifier(
        n_estimators=400,
        learning_rate=0.05,
        max_depth=5,
        scale_pos_weight=scale_pos_weight,
        tree_method="hist",
        eval_metric="aucpr",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )

rows = []
for name, model in candidates.items():
    auc_scores = cross_val_score(model, X, y, cv=cv, scoring="roc_auc", n_jobs=-1)
    prc_scores = cross_val_score(model, X, y, cv=cv, scoring="average_precision", n_jobs=-1)
    f1_scores  = cross_val_score(model, X, y, cv=cv, scoring="f1", n_jobs=-1)
    rows.append({
        "model": name,
        "roc_auc_mean": auc_scores.mean(),
        "roc_auc_std":  auc_scores.std(),
        "pr_auc_mean":  prc_scores.mean(),
        "pr_auc_std":   prc_scores.std(),
        "f1_mean":      f1_scores.mean(),
        "f1_std":       f1_scores.std(),
    })
    print(f"{name:15s}  ROC-AUC={auc_scores.mean():.3f}±{auc_scores.std():.3f}  "
          f"PR-AUC={prc_scores.mean():.3f}±{prc_scores.std():.3f}  "
          f"F1@0.5={f1_scores.mean():.3f}±{f1_scores.std():.3f}")

cv_results = pd.DataFrame(rows).set_index("model").sort_values("pr_auc_mean", ascending=False)
print("\nranked by PR-AUC (F1@0.5 is at the default threshold — see §5.2 for tuned F1):")
cv_results.round(3)

class counts: pos=63, neg=1461, scale_pos_weight=23.190
logreg           ROC-AUC=0.908±0.010  PR-AUC=0.403±0.089  F1@0.5=0.335±0.026
random_forest    ROC-AUC=0.880±0.021  PR-AUC=0.434±0.113  F1@0.5=0.387±0.124
lightgbm         ROC-AUC=0.925±0.019  PR-AUC=0.482±0.102  F1@0.5=0.442±0.113
xgboost          ROC-AUC=0.932±0.027  PR-AUC=0.465±0.071  F1@0.5=0.477±0.062

ranked by PR-AUC (F1@0.5 is at the default threshold — see §5.2 for tuned F1):


,roc_auc_mean,roc_auc_std,pr_auc_mean,pr_auc_std,f1_mean,f1_std
model,,,,,,
lightgbm,0.925,0.019,0.482,0.102,0.442,0.113
xgboost,0.932,0.027,0.465,0.071,0.477,0.062
random_forest,0.880,0.021,0.434,0.113,0.387,0.124
logreg,0.908,0.010,0.403,0.089,0.335,0.026


### 3.2 Pick the winner by PR-AUC

In [67]:
winner_name = cv_results.index[0]
print(f"winner: {winner_name}")

winner: lightgbm


---
## 4 · Hyperparameter tuning

Small, targeted grid for whichever baseline won. Scoring = PR-AUC (`average_precision`) because the class imbalance makes it a more honest objective than ROC-AUC.

In [68]:
if winner_name == "random_forest":
    base = RandomForestClassifier(class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE)
    param_grid = {
        "n_estimators": [300, 600],
        "max_depth": [None, 8, 16],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", 0.5],
    }
elif winner_name == "lightgbm":
    base = LGBMClassifier(
        class_weight="balanced", n_jobs=-1,
        random_state=RANDOM_STATE, verbose=-1,
    )
    param_grid = {
        "n_estimators": [200, 400, 800],
        "learning_rate": [0.03, 0.05, 0.1],
        "num_leaves": [15, 31, 63],
        "min_child_samples": [10, 20, 40],
    }
elif winner_name == "xgboost":
    base = XGBClassifier(
        scale_pos_weight=scale_pos_weight,
        tree_method="hist", eval_metric="aucpr",
        n_jobs=-1, random_state=RANDOM_STATE,
    )
    param_grid = {
        "n_estimators": [200, 400, 800],
        "learning_rate": [0.03, 0.05, 0.1],
        "max_depth": [3, 5, 7],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0],
    }
else:  # logreg
    base = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=5000, class_weight="balanced", random_state=RANDOM_STATE)),
    ])
    param_grid = {
        "clf__C": [0.01, 0.1, 1.0, 10.0],
        "clf__penalty": ["l2"],
    }

grid = GridSearchCV(
    base,
    param_grid=param_grid,
    scoring="average_precision",
    cv=cv,
    n_jobs=-1,
    refit=True,
    verbose=0,
)
grid.fit(X, y)
print("best PR-AUC:", round(grid.best_score_, 4))
print("best params:", grid.best_params_)
best_model = grid.best_estimator_

best PR-AUC: 0.583
best params: {'learning_rate': 0.03, 'min_child_samples': 40, 'n_estimators': 200, 'num_leaves': 31}


---
## 5 · Final evaluation

### 5.1 Cross-validated metrics

ROC-AUC, PR-AUC, and F1 @ 0.5 from the same 5-fold CV used for model selection. With ~63 total positives, per-fold means with their std deviations are the honest summary.

In [69]:
auc_scores = cross_val_score(best_model, X, y, cv=cv, scoring="roc_auc", n_jobs=-1)
prc_scores = cross_val_score(best_model, X, y, cv=cv, scoring="average_precision", n_jobs=-1)
f1_scores  = cross_val_score(best_model, X, y, cv=cv, scoring="f1", n_jobs=-1)

print(f"ROC-AUC : {auc_scores.mean():.3f} ± {auc_scores.std():.3f}")
print(f"PR-AUC  : {prc_scores.mean():.3f} ± {prc_scores.std():.3f}")
print(f"F1 @0.5 : {f1_scores.mean():.3f} ± {f1_scores.std():.3f}")

ROC-AUC : 0.951 ± 0.017
PR-AUC  : 0.583 ± 0.100
F1 @0.5 : 0.494 ± 0.041


### 5.2 Tune the decision threshold

Refit `best_model` on the full dataset (it's already refit by GridSearchCV), read `predict_proba` for every segment, sweep thresholds across the PR curve, and pick the one that maximises F1. That threshold becomes the operating cut-off used in §6.

In [70]:
# best_model is already refit on X, y by GridSearchCV(refit=True)
proba_all = best_model.predict_proba(X)[:, 1]

prec, rec, thr = precision_recall_curve(y, proba_all)
f1_curve = 2 * prec * rec / (prec + rec + 1e-12)
best_idx = int(np.nanargmax(f1_curve[:-1]))
best_threshold = float(thr[best_idx])

pred_tuned = (proba_all >= best_threshold).astype(int)
print(f"optimal F1 threshold: {best_threshold:.3f}  (F1={f1_curve[best_idx]:.3f})")
print("\nclassification report at tuned threshold (refit on full data):")
print(classification_report(y, pred_tuned, digits=3))

cm = confusion_matrix(y, pred_tuned)
cm_df = pd.DataFrame(cm, index=["actual_0", "actual_1"], columns=["pred_0", "pred_1"])
print("\nconfusion matrix:")
print(cm_df)

optimal F1 threshold: 0.869  (F1=0.833)

classification report at tuned threshold (refit on full data):
              precision    recall  f1-score   support

           0      0.998     0.986     0.992      1461
           1      0.741     0.952     0.833        63

    accuracy                          0.984      1524
   macro avg      0.869     0.969     0.913      1524
weighted avg      0.987     0.984     0.985      1524


confusion matrix:
          pred_0  pred_1
actual_0    1440      21
actual_1       3      60


### 5.3 Feature importance

Tree-based models expose `feature_importances_` directly; logistic regression exposes standardised coefficients (absolute value ⇒ importance).

In [71]:
def feature_importances(model, feature_names):
    """Pull importances out of a Pipeline or a raw estimator."""
    estimator = model.named_steps["clf"] if hasattr(model, "named_steps") else model
    if hasattr(estimator, "feature_importances_"):
        vals = estimator.feature_importances_
        label = "importance"
    elif hasattr(estimator, "coef_"):
        vals = np.abs(estimator.coef_[0])
        label = "abs_coef"
    else:
        raise ValueError("model has no importances / coefficients")
    return pd.Series(vals, index=feature_names, name=label).sort_values(ascending=False)

fi = feature_importances(best_model, X_train.columns)
print("top-20 features:")
fi.head(20).to_frame()

top-20 features:


,importance
length_m,903
demand_vs_supply,481
road_max_nearest_m,224
road_mean_nearest_m,134
gap_severity,122
road_span_lat,62
nearest_grid_dist_deg,54
road_centroid_lat,51
road_total_refill_points,41
power_range_kw,39


---
## 6 · Predict optimal locations

Refit the tuned model on the **full dataset** (train + test) so predictions use every available row. Then score every segment, rank by probability, and export `optimal_locations.csv` with road centroid coordinates for mapping.

### 6.1 Refit on the full dataset

In [72]:
# best_model is already fit on the full dataset (§4 GridSearchCV refit=True, re-verified in §5.2).
proba_all = best_model.predict_proba(X)[:, 1]
pred_all  = (proba_all >= best_threshold).astype(int)
print(f"scored {len(proba_all)} segments; {pred_all.sum()} flagged as needs_station "
      f"(threshold={best_threshold:.3f})")

scored 1524 segments; 81 flagged as needs_station (threshold=0.869)


### 6.2 Rank & merge with segment metadata

In [73]:
# road_centroid_lat / road_centroid_lon survive FE (they're valid features).
coord_cols = [c for c in ["road_centroid_lat", "road_centroid_lon"] if c in X.columns]

ranked = (
    pd.concat(
        [
            segment_meta.reset_index(drop=True),
            X[coord_cols].reset_index(drop=True),
        ],
        axis=1,
    )
    .assign(
        prob_needs_station=proba_all,
        predicted_needs_station=pred_all,
        actual_is_underserved=y.values,
    )
    .sort_values("prob_needs_station", ascending=False)
    .reset_index(drop=True)
)
print("top-20 segments by predicted probability:")
ranked.head(20)

top-20 segments by predicted probability:


,segment_id,carretera,comunidad_autonoma,provincia,municipio,road_centroid_lat,road_centroid_lon,prob_needs_station,predicted_needs_station,actual_is_underserved
0,1025,N-401,08 - C.La Mancha,Ciudad Real,Puertollano,39.001170,-3.928287,0.991121,1,1
1,705,N-211,02 - Aragón,Teruel,Escucha,40.890687,-0.769281,0.990548,1,1
2,733,N-232A,02 - Aragón,Teruel,Alcañiz,41.166500,-0.267939,0.989595,1,1
3,734,N-232A,02 - Aragón,Teruel,Alcañiz,41.166500,-0.267939,0.989595,1,1
4,702,N-204,unknown,unknown,unknown,40.603296,-3.247541,0.989011,1,1
5,744,N-232A,02 - Aragón,Teruel,Alcañiz,41.166500,-0.267939,0.988985,1,1
6,737,N-232A,02 - Aragón,Teruel,Alcañiz,41.166500,-0.267939,0.988985,1,1
7,739,N-232A,02 - Aragón,Teruel,Alcañiz,41.166500,-0.267939,0.988985,1,1
8,731,N-232A,02 - Aragón,Teruel,Alcañiz,41.166500,-0.267939,0.988046,1,1
9,735,N-232A,02 - Aragón,Teruel,Alcañiz,41.166500,-0.267939,0.987972,1,1


### 6.3 Export `optimal_locations.csv`

Two outputs:
- **`optimal_locations.csv`** — *every* segment with its probability (audit trail / downstream use).
- **`optimal_locations_top.csv`** — only the segments predicted `1` at the optimal threshold (proposed station candidates).

Coordinates come from `road_centroid_lat/lon`; note that these are *road-level* centroids, so all segments on the same road share them. Downstream File 2 generation should refine to segment-specific coordinates.

In [74]:
out_all = f"{DATA_DIR}/optimal_locations.csv"
out_top = f"{DATA_DIR}/optimal_locations_top.csv"

ranked.to_csv(out_all, index=False)
ranked.query("predicted_needs_station == 1").to_csv(out_top, index=False)

print("wrote:")
print(f"  {out_all}  — {len(ranked)} rows (all segments ranked)")
print(f"  {out_top}  — {(ranked['predicted_needs_station']==1).sum()} rows (proposed stations)")

wrote:
  C:/Users/vldma/Datathon/Datasets/Modelling datasets/optimal_locations.csv  — 1524 rows (all segments ranked)
  C:/Users/vldma/Datathon/Datasets/Modelling datasets/optimal_locations_top.csv  — 81 rows (proposed stations)


---
## 7 · Handoff summary

- **Best model**: reported in §3 ranking + §4 tuning (`winner_name` + `grid.best_params_`).
- **Operating threshold**: `best_threshold` from §5.2 — F1-optimal on the held-out set.
- **Top-K segments** by `prob_needs_station` are the input to File 2 (proposed station locations).
- **Known limitation**: `road_centroid_lat/lon` is one point per *road*, not per *segment* — every segment on the same road currently shares coordinates. A downstream step (geometry interpolation along each segment's polyline) is required before File 2 can be published.
- **Not yet covered** (for competition compliance): `grid_status` assignment (`Sufficient | Moderate | Congested`), `distributor_network` tagging (`i-DE | Endesa | Viesgo`), interurban-only road filter (`^(A|AP|N)-`), and `estimated_demand_kw = n_chargers × 150 kW` — all belong in a separate output-generation notebook that builds `File 1.csv` / `File 2.csv` / `File 3.csv`.